In [ ]:
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ----------------------------------------------------------
# PASO 1: CONFIGURACIÓN DEL NAVEGADOR
# ----------------------------------------------------------

# Creamos el objeto de opciones para Chrome.
options = Options()

# Ejecuta el navegador en modo invisible.
options.add_argument("--headless")

# Evita algunos problemas de permisos en entornos Linux o contenedores.
options.add_argument("--no-sandbox")

# Reduce errores relacionados con memoria compartida en Docker.
options.add_argument("--disable-dev-shm-usage")

# Inicializamos el navegador Chrome con las opciones indicadas.
driver = webdriver.Chrome(options=options)


# ----------------------------------------------------------
# PASO 2: VARIABLES DE TRABAJO
# ----------------------------------------------------------

# Lista donde iremos acumulando todos los hoteles encontrados.
# Cada elemento será un diccionario con nombre y URL.
datos_totales = []

# Por ahora dejamos 1 página como prueba inicial.
paginas_objetivo = 1


try:
    # ------------------------------------------------------
    # PASO 3: ABRIR LA PÁGINA OBJETIVO
    # ------------------------------------------------------

    # Cargamos la página de hoteles en Chile dentro de TripAdvisor.
    driver.get("https://www.tripadvisor.cl/Hotels-g294291-Chile-Hotels.html")

    # Recorremos la cantidad de páginas definida.
    # En esta primera versión solo será 1 iteración.
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        # --------------------------------------------------
        # PASO 4: ESPERAR A QUE CARGUEN LOS ELEMENTOS
        # --------------------------------------------------

        # Esperamos hasta 15 segundos a que aparezcan enlaces que correspondan a hoteles.        
        #WebDriverWait(driver, 30).until(
        #    EC.presence_of_all_elements_located(
        #        (By.XPATH, "//a[contains(@href,'/Hotel_Review-')]")
        #    )
        #)

        time.sleep(8)

        hoteles = driver.find_elements(By.XPATH, "//a[contains(@href,'Hotel_Review')]")
        print("Cantidad de enlaces detectados:", len(hoteles))

        # --------------------------------------------------
        # PASO 5: CAPTURAR LOS ENLACES DE HOTELES
        # --------------------------------------------------

        # Obtenemos todos los elementos <a> que apunten a páginas
        # de hoteles.
        hoteles = driver.find_elements(By.XPATH, "//a[contains(@href,'/Hotel_Review-')]")

        # Creamos un conjunto para evitar guardar hoteles repetidos.
        # Esto es útil porque a veces en una misma página aparecen
        # enlaces duplicados al mismo hotel.
        vistos = set()

        # Recorremos cada enlace detectado.
        for h in hoteles:
            try:
                nombre = h.text.strip()
                enlace = h.get_attribute("href")
        
                if not nombre or not enlace:
                    continue
        
                nombre = re.sub(r"^\d+\.\s*", "", nombre)
        
                clave = (nombre, enlace)
                if clave in vistos:
                    continue
                vistos.add(clave)
        
                datos_totales.append({
                    "hotel": nombre,
                    "url": enlace
                })
        
            except Exception:
                continue

    # ------------------------------------------------------
    # PASO 6: RESUMEN DEL PROCESO
    # ------------------------------------------------------

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

    # ------------------------------------------------------
    # PASO 7: GUARDAR LOS DATOS EN CSV
    # ------------------------------------------------------

    # Solo intentamos guardar si realmente se capturaron datos.
    if datos_totales:
        # Convertimos la lista de diccionarios en un DataFrame.
        df = pd.DataFrame(datos_totales)

        # Definimos el nombre del archivo de salida.
        nombre_archivo = "tripadvisor_hoteles_chile.csv"

        # Guardamos el DataFrame como CSV.
        # index=False evita agregar una columna numérica extra.
        # encoding="utf-8-sig" ayuda a que Excel abra mejor tildes y eñes.
        df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

        print(f"Archivo CSV generado correctamente: {nombre_archivo}")

        # Mostramos las primeras filas para verificar el resultado.
        print(df.head())
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print("Tipo de error:", type(e).__name__)
    print("Detalle del error:", repr(e))

finally:
    # ------------------------------------------------------
    # PASO 8: CIERRE SEGURO DEL NAVEGADOR
    # ------------------------------------------------------

    # Cerramos el navegador para liberar memoria y procesos.
    driver.quit()

In [8]:
import pandas as pd
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import time

options = Options()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

datos_totales = []

try:
    driver.get("https://www.tripadvisor.cl/Hotels-g294291-Chile-Hotels.html")

    print("URL actual:", driver.current_url)
    print("Título actual:", driver.title)
    print(driver.page_source[:1000])

    print("--- Procesando Página 1 ---")

    time.sleep(8)

    hoteles = driver.find_elements(By.XPATH, "//a[contains(@href,'Hotel_Review')]")
    print("Cantidad de enlaces detectados:", len(hoteles))

    vistos = set()

    for h in hoteles:
        try:
            nombre = h.text.strip()
            enlace = h.get_attribute("href")

            if not nombre or not enlace:
                continue

            nombre = re.sub(r"^\d+\.\s*", "", nombre)

            clave = (nombre, enlace)
            if clave in vistos:
                continue
            vistos.add(clave)

            datos_totales.append({
                "hotel": nombre,
                "url": enlace
            })

        except Exception:
            continue

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

    if datos_totales:
        df = pd.DataFrame(datos_totales)
        nombre_archivo = "tripadvisor_hoteles_chile.csv"
        df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")
        print(f"Archivo CSV generado correctamente: {nombre_archivo}")
        print(df.head())
    else:
        print("No hay datos para guardar.")

except Exception as e:
    print("Tipo de error:", type(e).__name__)
    print("Detalle del error:", repr(e))

finally:
    driver.quit()

URL actual: https://www.tripadvisor.cl/Hotels-g294291-Chile-Hotels.html
Título actual: tripadvisor.cl
<html lang="en"><head><title>tripadvisor.cl</title><style>#cmsg{animation: A 1.5s;}@keyframes A{0%{opacity:0;}99%{opacity:0;}100%{opacity:1;}}</style><meta name="viewport" content="width=device-width, initial-scale=1.0"></head><body style="margin:0"><script data-cfasync="false">var dd={'rt':'c','cid':'AHrlqAAAAAMAXe1SMftwUpAAklN3Cg==','hsh':'2F05D671381DB06BEE4CC52C7A6FD3','t':'bv','qp':'','s':46694,'e':'2a89be6087c86741fb7a9bf095ef3787c1d2e513569bf3ac230c4d944883b476ab78f4bdf9616e13e8a220ea334f2e3f','host':'geo.captcha-delivery.com','cookie':'89Uk4ncxueDDxrLd7dQ1lQQeLKMSni6mE3Taj7ZyVwxp9OB412sxLw~1x5wsfVqhpwUkbra7~7xKiomLhwNXULymN0B9SY_mCtWp~TFl8rITykx5H4RoSzPsUar4JCBo'}</script><script data-cfasync="false" src="https://ct.captcha-delivery.com/c.js"></script><iframe src="https://geo.captcha-delivery.com/captcha/?initialCid=AHrlqAAAAAMAXe1SMftwUpAAklN3Cg%3D%3D&amp;hash=2F05D671381DB06B

In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

options = Options()
# sin headless
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
driver.get("https://www.google.com")
input("Si ves el navegador abierto, presiona Enter aquí...")
driver.quit()

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x5e6e9eccca6a <unknown>
#1 0x5e6e9e6dbab5 <unknown>
#2 0x5e6e9e71866b <unknown>
#3 0x5e6e9e713339 <unknown>
#4 0x5e6e9e76353e <unknown>
#5 0x5e6e9e762c2c <unknown>
#6 0x5e6e9e721cbf <unknown>
#7 0x5e6e9e722a81 <unknown>
#8 0x5e6e9ec92a64 <unknown>
#9 0x5e6e9ec95951 <unknown>
#10 0x5e6e9ec7f21e <unknown>
#11 0x5e6e9ec9651e <unknown>
#12 0x5e6e9ec65be0 <unknown>
#13 0x5e6e9ecb99b8 <unknown>
#14 0x5e6e9ecb9b88 <unknown>
#15 0x5e6e9eccb4de <unknown>
#16 0x7c0d44151ac3 <unknown>
